# 1.0 - Análisis exploratorio de datos

Exploración inicial del dataset **House Prices - Advanced Regression Techniques** (Kaggle).

- **Objetivo:** entender la estructura de los datos, la variable objetivo `SalePrice`, los valores faltantes y las relaciones más relevantes.
- **Datos:** `data/raw/train.csv` y `data/raw/test.csv`.

## 1. Configuración

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from proyecto_pd.config import FIGURES_DIR, RAW_DATA_DIR

pd.set_option("display.max_columns", 100)
plt.style.use("ggplot")

## 2. Carga de datos

In [ ]:
train = pd.read_csv(RAW_DATA_DIR / "train.csv")
test = pd.read_csv(RAW_DATA_DIR / "test.csv")

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

In [ ]:
train.head()

## 3. Estructura general

In [ ]:
train.info()

In [ ]:
numericas = train.select_dtypes(include="number").columns.drop(["Id", "SalePrice"])
categoricas = train.select_dtypes(exclude="number").columns

print(f"Variables numéricas:   {len(numericas)}")
print(f"Variables categóricas: {len(categoricas)}")

In [ ]:
train[numericas].describe().T

## 4. Valores faltantes

In [ ]:
faltantes = pd.DataFrame({
    "train_%": train.isna().mean() * 100,
    "test_%": test.isna().mean() * 100,
})
faltantes = faltantes[(faltantes > 0).any(axis=1)].sort_values("train_%", ascending=False)
faltantes.round(2)

In [ ]:
ax = faltantes["train_%"].plot.barh(figsize=(8, 8))
ax.invert_yaxis()
ax.set_xlabel("% de valores faltantes (train)")
ax.set_title("Valores faltantes por variable")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "faltantes_train.png")
plt.show()

> Según `data_description.txt`, en varias variables (`PoolQC`, `Alley`, `Fence`, `FireplaceQu`, `GarageType`, `BsmtQual`, …) el faltante significa **"no existe"** (sin piscina, sin garaje, etc.), no un dato perdido.

## 5. Variable objetivo: `SalePrice`

In [ ]:
print(train["SalePrice"].describe())
print(f"\nAsimetría: {train['SalePrice'].skew():.2f}")
print(f"Curtosis:  {train['SalePrice'].kurt():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train["SalePrice"].plot.hist(bins=50, ax=axes[0], title="SalePrice")
np.log1p(train["SalePrice"]).plot.hist(bins=50, ax=axes[1], title="log(1 + SalePrice)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribucion_saleprice.png")
plt.show()

## 6. Correlación con las variables numéricas

In [ ]:
correlaciones = train[numericas].corrwith(train["SalePrice"]).sort_values(ascending=False)
correlaciones.head(15)

In [ ]:
top = correlaciones.head(10).index
corr_top = train[list(top) + ["SalePrice"]].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_top, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_top)), corr_top.columns, rotation=90)
ax.set_yticks(range(len(corr_top)), corr_top.columns)
fig.colorbar(im)
ax.set_title("Correlación entre las 10 variables más relacionadas con SalePrice")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlacion_top10.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, correlaciones.head(6).index):
    ax.scatter(train[col], train["SalePrice"], alpha=0.4, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel("SalePrice")
plt.tight_layout()
plt.show()

### 6.1 Separación de variables numéricas: discretas y continuas

Clasifica las variables numéricas según la cantidad de valores distintos: si tienen **15 o menos** se consideran **discretas** (conteos, calificaciones, códigos, mes/año); el resto son **continuas**. `PoolArea` se deja como continua porque es un área en pies², aunque tenga pocos valores distintos (casi ninguna casa tiene piscina).

In [ ]:
UMBRAL_DISCRETAS = 15

discretas = [
    col for col in numericas
    if train[col].nunique() <= UMBRAL_DISCRETAS and col != "PoolArea"
]
continuas = [col for col in numericas if col not in discretas]

print(f"Discretas ({len(discretas)}): {discretas}\n")
print(f"Continuas ({len(continuas)}): {continuas}")

### 6.2 Scatterplots de las variables continuas vs `SalePrice`

Recorre en un ciclo las variables continuas y grafica cada una contra `SalePrice`, mostrando en el título su correlación de Pearson (`r`). Al final crea un DataFrame con las correlaciones ordenadas de mayor a menor.

In [ ]:
n_cols = 4
n_filas = int(np.ceil(len(continuas) / n_cols))
fig, axes = plt.subplots(n_filas, n_cols, figsize=(5 * n_cols, 4 * n_filas))

resultados = []
for ax, col in zip(axes.flat, continuas):
    r = train[col].corr(train["SalePrice"])
    resultados.append({"variable": col, "correlacion": r, "correlacion_abs": abs(r)})

    ax.scatter(train[col], train["SalePrice"], alpha=0.4, s=10)
    ax.set_title(f"{col} (r = {r:.2f})")
    ax.set_xlabel(col)
    ax.set_ylabel("SalePrice")

# Ocultar los ejes que sobran en la última fila
for ax in axes.flat[len(continuas):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "scatter_continuas_vs_saleprice.png")
plt.show()

df_correlaciones_continuas = (
    pd.DataFrame(resultados)
    .sort_values("correlacion", ascending=False)
    .reset_index(drop=True)
)
df_correlaciones_continuas

### 6.3 Boxplots de las variables discretas vs `SalePrice`

Recorre en un ciclo las variables discretas y dibuja un boxplot de `SalePrice` para cada valor de la variable (eje X = variable, eje Y = target). Encima se muestran los puntos individuales con un *stripplot* (con un poco de dispersión horizontal) para ver cuántas casas hay en cada categoría y cómo se distribuyen.

In [ ]:
n_cols = 4
n_filas = int(np.ceil(len(discretas) / n_cols))
fig, axes = plt.subplots(n_filas, n_cols, figsize=(5 * n_cols, 4 * n_filas))

for ax, col in zip(axes.flat, discretas):
    sns.boxplot(data=train, x=col, y="SalePrice", ax=ax, color="lightsteelblue", showfliers=False)
    sns.stripplot(data=train, x=col, y="SalePrice", ax=ax, color="black", size=2, alpha=0.3, jitter=0.25)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)

# Ocultar los ejes que sobran en la última fila
for ax in axes.flat[len(discretas):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "boxplot_discretas_vs_saleprice.png")
plt.show()

## 7. Variables categóricas

In [ ]:
train[categoricas].nunique().sort_values(ascending=False)

In [ ]:
orden = train.groupby("Neighborhood")["SalePrice"].median().sort_values().index

fig, ax = plt.subplots(figsize=(14, 5))
train.boxplot(column="SalePrice", by="Neighborhood", positions=range(len(orden)), ax=ax)
ax.set_xticks(range(len(orden)), orden, rotation=90)
ax.set_title("SalePrice por vecindario")
plt.suptitle("")
plt.tight_layout()
plt.show()

### 7.1 Boxplots de las variables categóricas vs `SalePrice` y razón de correlación (η)

Recorre en un ciclo las variables categóricas y dibuja un boxplot de `SalePrice` por categoría, con los puntos individuales encima (*stripplot*). Las categorías se ordenan por mediana de precio y los faltantes se muestran como la categoría `"Ausente"` (en la mayoría de estas variables el faltante significa que la casa no tiene esa característica).

En el título de cada gráfica se muestra la **razón de correlación η (eta)**, que mide qué tanto del precio se explica por la categoría:

$$\eta = \sqrt{\frac{\sum_{g} n_g\,(\bar{y}_g - \bar{y})^2}{\sum_{i} (y_i - \bar{y})^2}}$$

- Numerador: variación **entre** categorías (qué tan lejos está la media de cada categoría de la media general).
- Denominador: variación **total** del precio.
- Va de **0** (todas las categorías tienen la misma media → la variable no informa nada) a **1** (dentro de cada categoría el precio es constante → la variable lo explica por completo).

Al final crea un DataFrame con η ordenado de mayor a menor.

In [ ]:
def razon_correlacion(categorias: pd.Series, y: pd.Series) -> float:
    """Razón de correlación η entre una variable categórica y una numérica."""
    media_general = y.mean()
    grupos = y.groupby(categorias)
    ss_entre = (grupos.count() * (grupos.mean() - media_general) ** 2).sum()
    ss_total = ((y - media_general) ** 2).sum()
    return np.sqrt(ss_entre / ss_total)


n_cols = 4
n_filas = int(np.ceil(len(categoricas) / n_cols))
fig, axes = plt.subplots(n_filas, n_cols, figsize=(5 * n_cols, 4 * n_filas))

resultados = []
for ax, col in zip(axes.flat, categoricas):
    valores = train[col].fillna("Ausente")
    eta = razon_correlacion(valores, train["SalePrice"])
    resultados.append({"variable": col, "eta": eta, "n_categorias": valores.nunique()})

    orden = train["SalePrice"].groupby(valores).median().sort_values().index
    sns.boxplot(x=valores, y=train["SalePrice"], order=orden, ax=ax, color="lightsteelblue", showfliers=False)
    sns.stripplot(x=valores, y=train["SalePrice"], order=orden, ax=ax, color="black", size=2, alpha=0.3, jitter=0.25)
    ax.set_title(f"{col} (η = {eta:.2f})")
    ax.tick_params(axis="x", rotation=90)

# Ocultar los ejes que sobran en la última fila
for ax in axes.flat[len(categoricas):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "boxplot_categoricas_vs_saleprice.png")
plt.show()

df_eta_categoricas = (
    pd.DataFrame(resultados)
    .sort_values("eta", ascending=False)
    .reset_index(drop=True)
)
df_eta_categoricas

## 8. Selección de variables

Selecciona las **10 mejores variables de cada grupo** según su métrica de relación con `SalePrice`:

- **Continuas:** correlación de Pearson en valor absoluto (`correlacion_abs`, sección 6.2).
- **Discretas:** razón de correlación η, tratando cada valor como un grupo (igual que en los boxplots de la sección 6.3).
- **Categóricas:** razón de correlación η (sección 7.1).

El resultado se guarda en el arreglo `features`, que será la lista de variables para construir el modelo.

In [ ]:
TOP_N = 10

df_eta_discretas = (
    pd.DataFrame([
        {"variable": col, "eta": razon_correlacion(train[col], train["SalePrice"])}
        for col in discretas
    ])
    .sort_values("eta", ascending=False)
    .reset_index(drop=True)
)

top_continuas = (
    df_correlaciones_continuas
    .sort_values("correlacion_abs", ascending=False)
    .head(TOP_N)["variable"]
    .tolist()
)
top_discretas = df_eta_discretas.head(TOP_N)["variable"].tolist()
top_categoricas = df_eta_categoricas.head(TOP_N)["variable"].tolist()

features = top_continuas + top_discretas + top_categoricas

print(f"Continuas ({len(top_continuas)}):   {top_continuas}")
print(f"Discretas ({len(top_discretas)}):   {top_discretas}")
print(f"Categóricas ({len(top_categoricas)}): {top_categoricas}")
print(f"\nTotal de features: {len(features)}")
features

## 9. Conclusiones

_Por completar a partir de los resultados:_

-
-
- 